# Limpieza y textometria

## Carga de datos

In [ ]:
from utils import load_corpus

df = load_corpus()
print(df.shape)
df.head(3)

## Limpieza y normalización

El corpus separa toda la puntuación con espacios, incluidos los separadores de millar (sección 2). Sobre esa base se aplican tres criterios:
1. Normalizacion de texto (previa a la tokenización): Se normalizan valores anomalos y/o extraños encontrados.
2. Minúsculas (al tokenizar, vía token.lower_): se aplica después de segmentar, porque el tokenizador usa la capitalización en sus reglas.
3. Exclusión de puntuación (al tokenizar): se extraen los tokens que cumplen el criterio de palabra en lugar de borrar caracteres, de modo que letras no españolas como ç o ï se conservan.

No se elimina ningún documento ni se aplica filtro de longitud mínima (sección 2).

### Paso 1 - Normalización

In [ ]:
from utils import normalizar_texto

df["texto_rep"] = normalizar_texto(df["texto"])

n_antes = df["texto"].str.count(r"\b000\b").sum()
n_despues = df["texto_rep"].str.count(r"\b000\b").sum()
print(f"fragmento '000': {n_antes:,} → {n_despues:,}")

### Paso 2 y 3

In [ ]:
from collections import Counter

import spacy

nlp = spacy.blank("es")

freqs = Counter()
longitudes = []

for doc in nlp.pipe(df["texto_rep"], batch_size=200):
    toks = [t.lower_ for t in doc if not t.is_punct and not t.is_space]
    freqs.update(toks)
    longitudes.append(len(toks))

print(f"tokens: {sum(longitudes):,} | tipos: {len(freqs):,}")

## Exploración

### Metricas del corpus

In [ ]:
import numpy as np

n_docs = len(longitudes)
total = sum(longitudes)
arr = np.array(longitudes)

print(f"documentos: {n_docs:,}")
print(f"palabras: {total:,}")
print(f"promedio: {total / n_docs:.2f}")
print(f"mediana: {np.median(arr):.0f} | desv: {arr.std():.2f}")
print(
    f"min/P25/P75/max: {arr.min()} / {np.percentile(arr, 25):.0f} / {np.percentile(arr, 75):.0f} / {arr.max()}"
)

### Top 20

#### Con stop words

In [ ]:
top_con = freqs.most_common(20)
for i, (w, n) in enumerate(top_con, 1):
    print(f"{i:2}. {w:12} {n:>9,}")

#### Sin stop words (comparativo NLTK vs spaCy)

In [ ]:
from nltk.corpus import stopwords
from spacy.lang.es.stop_words import STOP_WORDS

sw_nltk = set(stopwords.words("spanish"))
sw_spacy = set(STOP_WORDS)

top_nltk = [(w, n) for w, n in freqs.most_common() if w not in sw_nltk][:20]
top_spacy = [(w, n) for w, n in freqs.most_common() if w not in sw_spacy][:20]

for i, ((w1, n1), (w2, n2)) in enumerate(zip(top_nltk, top_spacy), 1):
    print(f"{i:2}. {w1:12} {n1:>8,}   |   {w2:12} {n2:>8,}")

In [ ]:
print("pm:", freqs["pm"], "| am:", freqs["am"])
print("pm en spacy:", "pm" in sw_spacy, "| am en spacy:", "am" in sw_spacy)

In [ ]:
tot = sum(freqs.values())
tipos = len(freqs)

for nombre, sw in [("nltk", sw_nltk), ("spacy", sw_spacy)]:
    n_tok = sum(n for w, n in freqs.items() if w not in sw)
    n_tip = sum(1 for w in freqs if w not in sw)
    print(
        f"{nombre:6} lista={len(sw):3}  tokens={n_tok:>11,} ({n_tok / tot:5.1%})  tipos={n_tip:>8,} ({n_tip / tipos:5.1%})"
    )

print(f"{'total':6} {'':7}  tokens={tot:>11,}          tipos={tipos:>8,}")

In [ ]:
anios = df["fecha"].dt.year
cum = anios.value_counts().sort_index().cumsum() / len(df)
cortes = [int(cum[cum >= q].index[0]) for q in (0.25, 0.50, 0.75)]
print("cortes:", cortes)
print(cum.round(3).to_string())

In [ ]:
from collections import defaultdict


def bloque(a: int) -> str:
    c1, c2, c3 = cortes
    if a <= c1:
        return f"1980–{c1}"
    if a <= c2:
        return f"{c1 + 1}–{c2}"
    if a <= c3:
        return f"{c2 + 1}–{c3}"
    return f"{c3 + 1}–2011"


arr_anio = anios.to_numpy()
arr_fuente = df["fuente"].to_numpy()

freqs = Counter()
longitudes = []
por_fuente = defaultdict(Counter)
por_decada = defaultdict(Counter)
por_bloque = defaultdict(Counter)

for i, doc in enumerate(nlp.pipe(df["texto_rep"], batch_size=200)):
    toks = [t.lower_ for t in doc if not t.is_punct and not t.is_space]
    freqs.update(toks)
    longitudes.append(len(toks))
    por_fuente[arr_fuente[i]].update(toks)
    por_decada[(arr_anio[i] // 10) * 10].update(toks)
    por_bloque[bloque(arr_anio[i])].update(toks)

print(f"tokens: {sum(longitudes):,} | tipos: {len(freqs):,}")

In [ ]:
import pandas as pd


def top_grupo(counters: dict, n: int = 15, sw: set = sw_nltk) -> pd.DataFrame:
    """Frecuencia por cada 10.000 tokens, excluyendo stopwords."""
    out = {}
    for k in sorted(counters, key=str):
        c = counters[k]
        tot = sum(c.values())
        top = [(w, v * 10_000 / tot) for w, v in c.most_common(300) if w not in sw][:n]
        out[k] = [f"{w} ({v:.1f})" for w, v in top]
    return pd.DataFrame(out)


top_grupo(por_fuente)

In [ ]:
print(nlp("$")[0].is_punct, nlp("$")[0].is_currency)

In [ ]:
top_grupo(por_decada)

In [ ]:
top_grupo(por_bloque)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

from utils import results_dir

RESULTS = results_dir()


def barras(counter, sw, titulo, archivo, n=20, mostrar=True):
    datos = [(w, v) for w, v in counter.most_common(300) if w not in sw][:n]
    palabras = [w for w, _ in datos][::-1]
    valores = [v for _, v in datos][::-1]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(palabras, valores, color="#444")
    ax.set_xlabel("Frecuencia")
    ax.set_title(titulo)
    plt.tight_layout()
    fig.savefig(RESULTS / f"{archivo}.png", dpi=300, bbox_inches="tight")
    if mostrar:
        plt.show()
    else:
        plt.close(fig)


def nube(counter, sw, archivo, n=200, mostrar=True):
    datos = dict([(w, v) for w, v in counter.most_common(2000) if w not in sw][:n])
    wc = WordCloud(
        width=1600,
        height=900,
        background_color="white",
        colormap="viridis",
        random_state=0,
    ).generate_from_frequencies(datos)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    plt.tight_layout()
    fig.savefig(RESULTS / f"{archivo}.png", dpi=300, bbox_inches="tight")
    if mostrar:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# Principales (candidatas al documento)
barras(freqs, set(), "Top 20 con stopwords", "top20_con_sw")
barras(freqs, sw_nltk, "Top 20 sin stopwords (NLTK)", "top20_sin_sw_nltk")
barras(freqs, sw_spacy, "Top 20 sin stopwords (spaCy)", "top20_sin_sw_spacy")
nube(freqs, sw_nltk, "nube_global_nltk")

# Anexo: por fuente
for f, c in por_fuente.items():
    slug = f.replace("www.", "").replace(".com", "")
    barras(c, sw_nltk, f"Top 20 — {slug}", f"top20_{slug}", mostrar=False)
    nube(c, sw_nltk, f"nube_{slug}", mostrar=False)

for d, c in sorted(por_decada.items()):
    barras(c, sw_nltk, f"Top 20 — {d}s", f"top20_decada_{d}", mostrar=False)

for b, c in sorted(por_bloque.items()):
    barras(
        c,
        sw_nltk,
        f"Top 20 — {b}",
        f"top20_bloque_{b.replace('–', '-')}",
        mostrar=False,
    )

print(len(list(RESULTS.glob("*.png"))), "figuras")

In [ ]:
import json

import numpy as np

arr = np.array(longitudes)

stats = {
    "documentos": len(longitudes),
    "tokens": int(arr.sum()),
    "tipos": len(freqs),
    "promedio": float(arr.mean()),
    "mediana": float(np.median(arr)),
    "desviacion": float(arr.std(ddof=1)),
    "minimo": int(arr.min()),
    "p25": float(np.percentile(arr, 25)),
    "p75": float(np.percentile(arr, 75)),
    "maximo": int(arr.max()),
    "stopwords_nltk": len(sw_nltk),
    "stopwords_spacy": len(sw_spacy),
    "eliminado_nltk": float(
        sum(n for w, n in freqs.items() if w in sw_nltk) / arr.sum()
    ),
    "eliminado_spacy": float(
        sum(n for w, n in freqs.items() if w in sw_spacy) / arr.sum()
    ),
    "fragmento_000_antes": int(n_antes),
    "fragmento_000_despues": int(n_despues),
    "docs_con_url": int(
        df["texto_rep"].str.contains(r"https?://|www\s*\.", regex=True).sum()
    ),
    "cortes_bloques": cortes,
}

(RESULTS / "textometria_stats.json").write_text(
    json.dumps(stats, indent=2, ensure_ascii=False)
)

pd.DataFrame(freqs.most_common(20), columns=["palabra", "frecuencia"]).to_csv(
    RESULTS / "top20_con_sw.csv", index=False
)

for nombre, sw in [("nltk", sw_nltk), ("spacy", sw_spacy)]:
    top = [(w, n) for w, n in freqs.most_common(300) if w not in sw][:20]
    pd.DataFrame(top, columns=["palabra", "frecuencia"]).to_csv(
        RESULTS / f"top20_sin_sw_{nombre}.csv", index=False
    )

for nombre, grupos in [
    ("fuente", por_fuente),
    ("decada", por_decada),
    ("bloque", por_bloque),
]:
    top_grupo(grupos).to_csv(RESULTS / f"top_por_{nombre}.csv", index=False)

print(json.dumps(stats, indent=2, ensure_ascii=False))